# Week 07: Building and checking RAG — the case

**Start here:** The assistant cites P3 for “two extensions,” but P3 permits one. A citation is present. Is the answer supported?

**Your goal:** Connect retrieved evidence to an answer and abstain when support is absent.

Run the prepared cells from top to bottom. Predict the result before changing one input. These small demonstrations isolate an idea; they are not trained production models.

**Concepts to point to:** RAG, Citation support, No-answer case, Test split.

# Week 7: RAG practice and held-out evaluation

A small extractive stand-in for a RAG application. No live LLM is used. Compare retrieval settings on 20 practice questions; teacher-held final cases are separate.

## Step 1: corpus, access filter, retrieval and answer
Read each function. Notice that source access is checked before ranking.

In [ ]:
import re
POLICIES = [{'id': 'P1', 'text': 'Student travel costs including bus tickets are not reimbursed.', 'role': 'student', 'version': '2027-01'}, {'id': 'P2', 'text': 'Library loans may be extended once for seven days.', 'role': 'student', 'version': '2027-01'}, {'id': 'P3', 'text': 'Students may request one deadline extension within seven days of the original due date.', 'role': 'student', 'version': '2027-01'}, {'id': 'P4', 'text': 'Academic appeals must be filed within ten days of the result.', 'role': 'student', 'version': '2027-01'}, {'id': 'P5', 'text': 'Staff payroll changes require manager approval before any record is changed.', 'role': 'staff', 'version': '2027-01'}]
STOP={'a','an','the','is','are','to','of','my','can','i','what','how','for','in','and'}
def tokens(text):return set(re.findall(r'[a-z]+',text.lower()))-STOP
def rank(question,role='student',k=1,corpus=None):
    if not isinstance(question,str) or not question.strip():raise ValueError('Question must be nonempty text')
    if role not in ('student','staff'):raise ValueError('Unknown trusted role')
    if k not in (1,2):raise ValueError('k must be 1 or 2')
    corpus=POLICIES if corpus is None else corpus
    allowed=[p for p in corpus if p['role']=='student' or role=='staff']
    scored=sorted([(len(tokens(question)&tokens(p['text'])),p) for p in allowed],key=lambda x:x[0],reverse=True)
    return [p for score,p in scored if score>=2][:k]
def answer(question,role='student',k=1,corpus=None):
    found=rank(question,role,k,corpus)
    return {'status':'extract' if found else 'unsupported','text':' '.join(p['text'] for p in found),
            'sources':[p['id'] for p in found],'version':'2027-01'}
def safe_trace(request_id,result):
    return {'request_id':request_id,'status':result['status'],'sources':result['sources'],'version':result['version']}


## Step 2: compare k=1 and k=2
Recall averages only answerable cases. No-answer correctness is reported separately. Citation support still needs human checking for generated answers.

In [ ]:
practice=[('student travel costs', ['P1']), ('bus tickets reimbursed', ['P1']), ('travel costs including bus tickets', ['P1']), ('student travel reimbursed', ['P1']), ('library loans extended', ['P2']), ('loans extended once', ['P2']), ('library loans seven days', ['P2']), ('extended library loans', ['P2']), ('deadline extension within seven days', ['P3']), ('students request deadline extension', ['P3']), ('original due date extension', ['P3']), ('one deadline extension', ['P3']), ('academic appeals within ten days', ['P4']), ('appeals filed ten days', ['P4']), ('appeals result ten days', ['P4']), ('academic appeals filed', ['P4']), ('cafeteria lunch menu', []), ('sports club hours', []), ('staff payroll changes', []), ('library loans and deadline extension', ['P2', 'P3'])]
for k in [1,2]:
    recalls=[];no_answer=[]
    for q,expected in practice:
        result=answer(q,k=k);got=set(result['sources'])
        if expected:recalls.append(len(got&set(expected))/len(expected))
        else:no_answer.append(not got)
        print(k,q,'expected',expected,'observed',result['sources'])
    print('k=',k,'mean recall=',round(sum(recalls)/len(recalls),3),'no-answer correct=',sum(no_answer),'/',len(no_answer))
assert len(practice)==20


## Step 3: make one justified choice
Record chosen k, observed misses and a limitation. Do not inspect final_questions.json until your teacher releases it. A large k can add irrelevant evidence. Your final report must use fresh cases and a frozen setting.

## Explain and transfer

1. What did you change, and what happened? Replace a relevant passage with an irrelevant one and observe abstention.
2. Which diagram block produced the difference?
3. What is one limitation of the demonstration?
4. Fresh situation: A fresh question asks about a rule absent from all five policies. What should the assistant do?

**Evidence to save:** Twenty-case practice table; ten final questions remain sealed for assessment.